# Att bygga ett återanvändbart aktuariellt prissättningsbibliotek med PROC FCMP


## Sammanfattning

Ett skador- och olycksfallsförsäkringsbolag kapslar in sin centrala prissättningsmatematik — ren premie, kostnads-/riskpåslag, trovärdighetsblandning enligt metoden för begränsad fluktuation, samt diskonterad reservberäkning — som egna funktioner och en subrutin med flera returvärden i **PROC FCMP**. Det kompilerade biblioteket registreras via systemoptionen `CMPLIB=` och anropas sedan rad för rad från ett DATA-steg som prissätter en syntetisk portfölj på 100 försäkringar. Varje prissatt värde i detta notebook — trovärdighetsfaktorn `z`, den blandade rena premien, den debiterade premien och den nuvärdesberäknade skadereserven — produceras av dessa kompilerade rutiner, inte av inline-aritmetik. Resultatet: den implicita skadeprocenten hamnar på 60,5 % (landsbygd), 55,8 % (förort) och 63,6 % (stad) — bekvämt under 100 % i varje segment, vilket bekräftar att den påslagna premien täcker den förväntade skadan samtidigt som prissättningssteget förblir rent och granskningsbart.


## Datakällor

| Dataset | Rader | Beskrivning | Nyckelvariabler |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Syntetisk gällande skador-/olycksfallsförsäkringsportfölj genererad inline med `rand()` | `policy_id`, `region` (stad/förort/landsbygd), `years_insured`, `exposure` (fordonsår), `claim_count` (Poisson), `incurred_loss` (gammaskadebelopp x antal), `class_pure_prem` (manuell tariff per region) |

DATA-steget loopar över ett bredare `policy_id`-intervall, men denna miljö körs olicensierad, så utdata begränsas till de första **100 observationerna** — den prissatta boken nedan speglar dessa 100 försäkringar.


# Att bygga ett återanvändbart aktuariellt prissättningsbibliotek med PROC FCMP

Aktuarier upprepar samma beräkningar över prissättnings-, reserverings- och rapporteringsjobb: omvandla skador till en *ren premie*, tillämpa *kostnads- och riskpåslag* för att nå en debiterad premie, blanda en enskild försäkrings egen erfarenhet med en klasstariff via *trovärdighet*, och *diskontera* framtida kassaflöden till nuvärde. Att skriva om dessa formler i varje DATA-steg inbjuder till copy-paste-fel och gör ändringar av antaganden besvärliga.

**PROC FCMP** (SAS funktionskompilator) låter oss definiera varje formel en gång som en namngiven funktion eller subrutin, lagra de kompilerade rutinerna i ett bibliotek och sedan anropa dem som vilken inbyggd SAS-funktion som helst. I detta notebook kommer vi att:

1. Kompilera ett litet aktuariellt funktionsbibliotek med `PROC FCMP`.
2. Registrera det för sessionen med systemoptionen `CMPLIB=`.
3. Generera en syntetisk skador- och olycksfallsförsäkringsportfölj.
4. Prissätta varje försäkring genom att anropa våra egna funktioner och en subrutin med flera returvärden från ett enda DATA-steg.
5. Summera och tolka den prissatta portföljen.


## Steg 1 — Generera en syntetisk försäkringsportfölj

Vi simulerar en bok med gällande bilförsäkringar (de första 100 prissätts nedan under gränsen för olicensierat läge). Varje försäkring tillhör en tariffregion med sin egen manuella *rena premie* (förväntad skada per fordonsår). Antal skador följer en Poisson-process skalad med exponering, och skadebelopp följer en gammafördelning, så `incurred_loss` är en realistisk sammansatt (Poisson x gamma) skada. `years_insured` styr senare trovärdighetsvikten. Ett fast frö via `call streaminit` gör datan reproducerbar.


In [1]:
data portfolio;
    CALL streaminit(20260531);
    LÄNGD region $12;
    GÖR policy_id = 10001 TILL 12000;
        /* Tilldela tariffregion */
        u = rand('uniform');
        OM u < 0.40 SÅ GÖR; region = 'stad';    base_pp = 820; lambda = 0.18; SLUT;
        ANNARS OM u < 0.75 SÅ GÖR; region = 'förort'; base_pp = 540; lambda = 0.11; SLUT;
        ANNARS GÖR; region = 'landsbygd';    base_pp = 360; lambda = 0.07; SLUT;

        /* Innehavstid (försäkringsår) och exponering (fordonsår) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Sammansatt skadeprocess: Poisson-frekvens, gammaskadebelopp */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        GÖR c = 1 TILL claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        SLUT;
        incurred_loss = round(incurred_loss, 0.01);

        /* Manuell klass-ren premie för denna försäkrings exponering */
        class_pure_prem = round(base_pp * exposure, 0.01);

        UTDATA;
    SLUT;
    BEHÅLL policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
    ETIKETT
        policy_id       = 'Försäkringsnummer'
        region          = 'Region'
        years_insured   = 'Försäkringsår'
        exposure        = 'Exponering (fordonsår)'
        claim_count     = 'Antal skador'
        incurred_loss   = 'Inträffad skada'
        class_pure_prem = 'Klassens rena premie';
KÖR;

PROCEDUR SKRIV data=portfolio(obs=8) noobs;
    TITEL 'De första 8 simulerade försäkringarna';
KÖR;


                                         De första 8 simulerade försäkringarna                                          

policy_id     region  years_insured  exposure  claim_count  incurred_loss  class_pure_prem
    10001  landsbygd              5      1.36            0              0            489.6
    10002  stad                   8      0.97            1        3432.28            795.4
    10003  stad                   2      1.53            2        7155.92           1254.6
    10004  förort                 9       2.4            0              0             1296
    10005  landsbygd              5      0.99            0              0            356.4
    10006  förort                 1      0.83            0              0            448.2
    10007  landsbygd              5      0.97            0              0            349.2
    10008  landsbygd              7      2.32            0              0            835.2

... 92 more observations (showing 8 of 100)




NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.44 seconds
  cpu   0.44 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Steg 2 — Kompilera det aktuariella funktionsbiblioteket

Nu notebookens kärna. `PROC FCMP` med `OUTLIB=work.actfuncs.pricing` kompilerar fyra funktioner och en subrutin till paketet `pricing` i datasetet `work.actfuncs`:

- **`pure_premium`** — observerad skada per enhet exponering (frekvens x skadebelopp kombinerat).
- **`credibility_z`** — trovärdighetsfaktor enligt begränsad fluktuation `Z = sqrt(n / (n + k))`, där `n` är försäkringens exponeringsår och `k` är en justeringskonstant.
- **`charged_premium`** — tillämpar ett proportionellt riskpåslag och en fast kostnadskvot på en skadekostnad för att nå den premie vi faktiskt debiterar.
- **`pv_reserve`** — nuvärdet av en framtida skadeutbetalning, `FV / (1+r)**t`, används för att diskontera skadereserver.
- **`blend_premium`** (en SUBROUTINE) — använder `OUTARGS` för att returnera *två* värden samtidigt: den trovärdighetsviktade rena premien och den trovärdighetsfaktor den använde, så att DATA-steget får båda i ett enda anrop.

Varje rutin avslutas med `ENDSUB`, och subrutinen namnger sina skrivbara argument med `OUTARGS`.


In [2]:
PROCEDUR fcmp outlib=work.actfuncs.pricing;

    /* Observerad ren premie: skadekostnad per enhet exponering */
    function pure_premium(loss, exposure);
        OM exposure <= 0 SÅ RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Trovärdighet enligt begränsad fluktuation: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        OM n_years <= 0 SÅ RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Debiterad premie = skadekostnad uppräknad för riskpåslag + kostnader */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        OM expense_ratio >= 1 SÅ RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Nuvärde av en framtida skadeutbetalning diskonterad t år till räntan r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Trovärdighetsblandning: returnerar den viktade rena premien OCH det använda Z */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

KÖR;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Steg 3 — Registrera biblioteket med CMPLIB=

Att kompilera rutinerna räcker inte; SAS måste få veta var de finns när ett senare DATA-steg eller en procedur refererar till ett namn den inte känner igen som inbyggt. Systemoptionen `CMPLIB=` pekar på datasetet (inte det tredelade paketnamnet) som innehåller den kompilerade koden. Efter detta `OPTIONS`-uttryck kan varje funktion och subrutin ovan anropas med namn.


In [3]:
ALTERNATIV cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Steg 4 — Prissätt portföljen med de egna funktionerna

Prissättnings-DATA-steget är nu nästan fritt från aritmetik — den aktuariella avsikten läses direkt ur funktionsnamnen. För varje försäkring:

1. Beräknar vi försäkringens egen observerade rena premie med `pure_premium`.
2. Anropar vi subrutinen `blend_premium` för att trovärdighetsvikta den mot den regionala klasstariffen, och får både den blandade skadekostnaden och trovärdighetsfaktorn `z` tillbaka via `OUTARGS`.
3. Räknar vi upp den blandade skadekostnaden till en debiterad premie med ett riskpåslag på 12 % och en kostnadskvot på 25 % via `charged_premium`.
4. Uppskattar vi den fortfarande öppna skadereserven som 35 % av försäkringens inträffade skada och diskonterar den tre år vid 4 % till nuvärde med `pv_reserve`.

Notera att subrutinens utdataargument (`blended_pp`, `z`) måste initieras före `CALL`. Reservens nuvärde varierar från försäkring till försäkring eftersom det styrs av varje försäkrings faktiska inträffade skada — skadefria försäkringar bär en nollreserv, så deras `reserve_pv` är också noll.


In [4]:
data rated;
    STÄLL_IN portfolio;

    /* 1. Försäkringens egen skadeerfarenhet som ren premie */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Trovärdighetsvikta egen erfarenhet mot klasstariffen.
          k = 4 exponeringsår för nästan full trovärdighet. */
    blended_pp = .;
    z = .;
    CALL blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Omvandla blandad skadekostnad (per fordonsår) till debiterad premie */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Utestående skadereserv = 35 % av inträffad skada, förväntas
          regleras om 3 år; diskontera den till nuvärde vid 4 %. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    BEHÅLL policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
    ETIKETT
        own_pp       = 'Egen ren premie'
        blended_pp   = 'Blandad ren premie'
        z            = 'Trovärdighetsfaktor Z'
        premium      = 'Premie'
        case_reserve = 'Skadereserv'
        reserve_pv   = 'Nuvärde reserv';
KÖR;

PROCEDUR SKRIV data=rated(obs=10) noobs;
    TITEL 'Prissatt portfölj (första 10 försäkringarna)';
    VARIABEL policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
KÖR;


                                      Prissatt portfölj (första 10 försäkringarna)                                      

policy_id     region  years_insured  exposure   own_pp  blended_pp      z  premium  reserve_pv
    10001  landsbygd              5      1.36        0       91.67  0.745   186.18           0
    10002  stad                   8      0.97  3538.43     3039.59  0.816  4402.95     1067.95
    10003  stad                   2      1.53  4677.07     3046.88  0.577  6961.51     2226.55
    10004  förort                 9       2.4        0       90.69  0.832   325.03           0
    10005  landsbygd              5      0.99        0       91.67  0.745   135.52           0
    10006  förort                 1      0.83        0       298.5  0.447   369.98           0
    10007  landsbygd              5      0.97        0       91.67  0.745   132.79           0
    10008  landsbygd              7      2.32        0       72.82  0.798   252.29           0
    10009  stad        


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Steg 5 — Summera den prissatta boken

Eftersom varje försäkring prissattes genom samma kompilerade bibliotek kan vi rulla upp portföljen per region. Vi redovisar medeldebiterad premie, medeltrovärdighetsfaktor, total inträffad skada och den totala nuvärdesberäknade skadereserven, och härleder sedan den implicita skadeprocenten per segment för att se om den påslagna premien täcker den förväntade skadan.


In [5]:
PROCEDUR MEDELVÄRDEN data=rated mean sum maxdec=2 nonobs;
    KLASS region;
    VARIABEL premium z incurred_loss reserve_pv;
    TITEL 'Portföljsammanfattning per tariffregion';
KÖR;

/* Implicit skadeprocent = inträffad skada / debiterad premie, plus den
   nuvärdesberäknade reserven segmentet bär, per region. */
PROCEDUR SQL;
    TITEL 'Implicit skadeprocent och reservnuvärde per region';
    VÄLJ region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     format=dollar12.2,
           sum(premium)                          AS total_premium  format=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     format=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve format=dollar12.2
    FROM rated
    GROUP EFTER region
    ORDER EFTER region;
QUIT;
TITEL;


                                        Portföljsammanfattning per tariffregion                                         

                                                  The MEANS Procedure

                                           Analysis Variable : premium Premie

        Region              Mean            Sum
        ---------------------------------------
        förort            813.04       34147.74
        landsbygd         476.61       11438.62
        stad             1987.17       67563.93
        ---------------------------------------

                                      Analysis Variable : z Trovärdighetsfaktor Z

        Region              Mean            Sum
        ---------------------------------------
        förort              0.68          28.36
        landsbygd           0.71          17.14
        stad                0.70          23.90
        ---------------------------------------

                                   Analysis Variable : incurred_los


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Att tolka resultaten

Prissättnings-DATA-steget skriver aldrig ut en enda diskonterings- eller trovärdighetsformel inline — det anropar bara `pure_premium`, `blend_premium`, `charged_premium` och `pv_reserve`. Det är vinsten med **PROC FCMP**: de aktuariella antagandena lever i ett enda kompilerat bibliotek som kan enhetstestas, versionshanteras och återanvändas över prissättnings-, reserverings- och rapporteringsjobb. Att ändra trovärdighetskonstanten `k`, riskpåslaget eller kostnadskvoten är en enda ändring i biblioteket, inte en jakt genom varje program.

Vid läsning av den prissatta boken och den regionala sammanställningen:

- **Trovärdighet (`z`)** stiger med `years_insured`, precis som formeln för begränsad fluktuation `Z = sqrt(n / (n + k))` föreskriver. Bland de första tio försäkringarna når den ettåriga förortsförsäkringen (10006) `z = 0,447`, den tvååriga stadsförsäkringen (10003) `z = 0,577`, medan den nioåriga förortsförsäkringen (10004) når `z = 0,832`. Försäkringar med tunn erfarenhet dras mot den regionala klasstariffen; långvariga lutar sig mot sina egna skador.
- **Blandning i praktiken:** skadefria försäkringar (merparten av boken) har `own_pp = 0`, så `blend_premium` returnerar en `blended_pp` nära `(1 - z)` gånger klasstariffen — t.ex. hamnar försäkring 10001 (landsbygd, `z = 0,745`) på `91,67` mot en klasstariff på `360`/fordonsår. De två stadsförsäkringarna som faktiskt drabbades av skador, 10002 och 10003, drar i stället upp sin blandade skadekostnad mot sin egen höga erfarenhet (`3039,59` och `3046,88`).
- **Den debiterade premien** ligger över den blandade skadekostnaden eftersom `charged_premium` lägger till ett riskpåslag på 12 % och räknar upp för en kostnadskvot på 25 %, en fast multiplikator på `1,12 / 0,75 = 1,493` på skadekostnaden. Medeldebiterad premie ligger på `476,61` (landsbygd), `813,04` (förort) och `1 987,17` (stad).
- **Diskonterade reserver:** `pv_reserve` diskonterar varje försäkrings utestående skadereserv (35 % av inträffad skada) tre år vid 4 %, en nuvärdesfaktor på `0,889`. Skadefria försäkringar bär `reserve_pv = 0`; de två stadsskadorna bidrar med `1067,95` och `2226,55`. Sammanräknat håller boken en nuvärdesberäknad reserv på `$2 151,56` (landsbygd), `$5 932,52` (förort) och `$13 364,13` (stad).
- **Implicita skadeprocent** hamnar på 60,5 % (landsbygd), 55,8 % (förort) och 63,6 % (stad) — alla bekvämt under 100 %, så den påslagna premien täcker den förväntade skadan i varje segment. Stadssegmentet är hetast, drivet av sina två stora simulerade skador; en verklig tariffgenomgång skulle testa om denna signal håller i sig över fler skadeår innan den manuella tariffen justeras.

Subrutinen `blend_premium` visar också `OUTARGS`-idiomet för att returnera flera resultat från ett `CALL` — den blandade skadekostnaden och trovärdighetsfaktorn `z` kommer tillbaka tillsammans — vilket undviker separata funktionsanrop och håller den försäkringsvisa prissättningslogiken kompakt och granskningsbar.
